# welded_fatigue Workflow — openBIS Provisioning Notebook

PRD: `BAM_PRD_Workflow_FB72.md`

openBIS: 20.10.12.5 | pyBIS: 1.37.4

Generated: 2026-06-02

This notebook is the **one-time setup path**. Run it once to create the Space/Project/Collection hierarchy and the parent specimen Object(s) plus their ExperimentalStep children. For per-run data ingestion, use the parser package (see cells 13-14).

In [ ]:
from pybis import Openbis
import getpass

## ⚠ FILL ME IN before running

Set these variables in the next cell:
- `OPENBIS_URL` — your openBIS server URL
- `SPACE` — target space code
- `PROJECT_CODE` — target project code
- `COLLECTION_CODE` — target collection code

In [ ]:
OPENBIS_URL = "https://your-openbis-instance.example.com"  # TODO: replace
SPACE = "YOUR_SPACE"           # TODO: replace
PROJECT_CODE = "YOUR_PROJECT"  # TODO: replace
COLLECTION_CODE = "WELDED_FATIGUE_01"  # TODO: replace
SESSION_NAME = "welded_fatigue-provisioning"

In [ ]:
# First-run PAT bootstrap: log in with password ONCE, mint a Personal Access Token,
# log out, then re-establish the session using the PAT only.
o = Openbis(OPENBIS_URL, verify_certificates=True)
username = input("openBIS username: ")
password = getpass.getpass("Password (first run only — PAT will be saved): ")
o.login(username, password, save_token=True)
token = o.get_or_create_personal_access_token(sessionName=SESSION_NAME)
print(f"PAT created: {token[:8]}…  Save this in a password manager.")
o.logout()
# From now on, use PAT only:
o = Openbis(OPENBIS_URL, verify_certificates=True)
o.set_token(token, save_token=True)

In [ ]:
assert o.is_session_active(), "Session not active — re-run PAT bootstrap cell"
print("Connected to openBIS. Session active.")

## Before continuing: push the masterdata extension

The custom types referenced below (e.g. `SPECIMEN.WELDED_FATIGUE`, all `EXPERIMENTAL_STEP.*` CREATE types, and the new vocabularies `ISO_5817_FAT_CLASS`, `LOAD_LEVEL`, `FATIGUE_STOP_REASON`) MUST exist in openBIS BEFORE any `new_sample(type=...)` call below succeeds. Push them via the CLI:

```bash
pip install -e ../bam-masterdata
python -m bam_masterdata masterdata_sync --url $OPENBIS_URL --token <your-pat>
```

Or submit `generated/welded_fatigue/datamodel/` as a PR to `BAMresearch/bam-masterdata` and wait for it to be merged + deployed.

Note: `WELDING.WELD_TYPE` (used by `weld_geometry`) requires the EXTEND PR adding `BUTT_WELD` and `CRUCIFORM_WELD` to be merged before those values are accepted.

In [ ]:
# Create the Collection (an openBIS Experiment under DEFAULT_EXPERIMENT type).
project_id = f"/{SPACE}/{PROJECT_CODE}"
collection = o.new_experiment(
    code=COLLECTION_CODE,
    type="DEFAULT_EXPERIMENT",
    project=project_id,
)
collection.save()
COLLECTION_ID = f"{project_id}/{COLLECTION_CODE}"
print(f"Collection created: {collection.permId}  identifier={COLLECTION_ID}")

In [ ]:
# WeldedFatigueSpecimen — SPECIMEN.WELDED_FATIGUE  (CREATE, per gap-report)
# Properties per requirements.json: original_id, sheet_origin, weld_geometry (WELDING.WELD_TYPE),
#   iso_fat_class (ISO_5817_FAT_CLASS), load_level (LOAD_LEVEL).
specimen = o.new_sample(
    type="SPECIMEN.WELDED_FATIGUE",
    space=SPACE,
    experiment=COLLECTION_ID,
    props={
        # TODO: fill in property values per your specimen
        # "original_id": "...",          # Manufacturer ID // Herstellerbezeichnung
        # "sheet_origin": "...",         # Sheet origin // Herkunftsblech
        # "weld_geometry": "BUTT_WELD",  # or "CRUCIFORM_WELD" — pending WELDING.WELD_TYPE EXTEND PR
        # "iso_fat_class": "C56",        # or "B90" / "B125" (ISO_5817_FAT_CLASS)
        # "load_level": "HIGH",          # or "MEDIUM" / "LOW" (LOAD_LEVEL)
    },
)
specimen.save()
print(f"Specimen created: {specimen.permId}")

In [ ]:
# All 12 FB7.2 ExperimentalSteps (CREATE per gap report). Each links to the specimen as parent.

# 1) PreQualityCheckWeld — EXPERIMENTAL_STEP.PRE_QUALITY_CHECK_WELD  (§5.3.1)
pre_qc = o.new_sample(
    type="EXPERIMENTAL_STEP.PRE_QUALITY_CHECK_WELD",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: fill in per requirements.json
        # "initial_angular_distortion": 0.5,   # [°]
        # "initial_edge_misalignment": 0.2,    # [mm]
        # "was_straightened": False,
    },
)
pre_qc.save()
print(f"PreQualityCheckWeld created: {pre_qc.permId}")

# 2) ChamferingGrinding — EXPERIMENTAL_STEP.CHAMFERING_GRINDING  (§5.3.2)
chamfer = o.new_sample(
    type="EXPERIMENTAL_STEP.CHAMFERING_GRINDING",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: fill in per requirements.json
        # "weld_modifications": "Reinforcement ground flush on tension side.",
    },
)
chamfer.save()
print(f"ChamferingGrinding created: {chamfer.permId}")

# 3) SpecimenRecording — EXPERIMENTAL_STEP.SPECIMEN_RECORDING  (§5.3.3)
spec_rec = o.new_sample(
    type="EXPERIMENTAL_STEP.SPECIMEN_RECORDING",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: object_link to GOM 3D-scanner Instrument (REUSE) — set via add_parents or property after creation
    },
)
spec_rec.save()
print(f"SpecimenRecording created: {spec_rec.permId}")

# 4) WeldAnalysis — EXPERIMENTAL_STEP.WELD_ANALYSIS  (§5.3.4)
weld_ana = o.new_sample(
    type="EXPERIMENTAL_STEP.WELD_ANALYSIS",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: fill in per requirements.json
        # "iso_fat_class": "C56",  # determined FAT class (ISO_5817_FAT_CLASS)
    },
)
weld_ana.save()
print(f"WeldAnalysis created: {weld_ana.permId}")

# 5) SeriesAssignment — EXPERIMENTAL_STEP.SERIES_ASSIGNMENT  (§5.3.5)
series = o.new_sample(
    type="EXPERIMENTAL_STEP.SERIES_ASSIGNMENT",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: documents temporal randomization and assignment to the test series
    },
)
series.save()
print(f"SeriesAssignment created: {series.permId}")

# 6) TestSetupGeometry — EXPERIMENTAL_STEP.TEST_SETUP_GEOMETRY  (§5.3.6)
setup_geo = o.new_sample(
    type="EXPERIMENTAL_STEP.TEST_SETUP_GEOMETRY",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: fill in per requirements.json
        # "avg_thickness": 5.0,  # [mm]
        # "avg_width": 40.0,     # [mm]
    },
)
setup_geo.save()
print(f"TestSetupGeometry created: {setup_geo.permId}")

# 7) MonitoringApplication — EXPERIMENTAL_STEP.MONITORING_APPLICATION  (§5.3.7)
monitoring = o.new_sample(
    type="EXPERIMENTAL_STEP.MONITORING_APPLICATION",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: fill in per requirements.json
        # "strain_gauge_type_ref": "<STRAIN_GAUGE_TYPE permId>",  # OBJECT link — CREATE_OPTIONAL
        # "consumable_type_ref": "<CONSUMABLE_TYPE permId>",      # OBJECT link — CREATE_OPTIONAL
        # "measured_dms_distance_weld": 5.0,   # [mm]
        # "measured_dms_distance_edge": 10.0,  # [mm]
    },
)
monitoring.save()
print(f"MonitoringApplication created: {monitoring.permId}")

# 8) AmplifierSettings — EXPERIMENTAL_STEP.AMPLIFIER_SETTINGS  (§5.3.8)
amp = o.new_sample(
    type="EXPERIMENTAL_STEP.AMPLIFIER_SETTINGS",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: fill in per requirements.json
        # "sampling_frequency": 100,            # [Hz]
        # "recording_mode": "Peak-Valley",
        # "testing_machine_ref": "<TESTING_MACHINE permId>",     # OBJECT link (REUSE)
        # "measuring_amplifier_ref": "<MEASURING_AMPLIFIER permId>",  # OBJECT link (REUSE)
    },
)
amp.save()
print(f"AmplifierSettings created: {amp.permId}")

# 9) InstallationStressMeasurement — EXPERIMENTAL_STEP.INSTALLATION_STRESS_MEASUREMENT  (§5.3.9)
inst_stress = o.new_sample(
    type="EXPERIMENTAL_STEP.INSTALLATION_STRESS_MEASUREMENT",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: captures bending moment at 0.0 kN before test start; results as attached file
    },
)
inst_stress.save()
print(f"InstallationStressMeasurement created: {inst_stress.permId}")

# 10) CyclicFatigueTest — EXPERIMENTAL_STEP.CYCLIC_FATIGUE_TEST  (§5.3.10)
cyclic_fatigue_test = o.new_sample(
    type="EXPERIMENTAL_STEP.CYCLIC_FATIGUE_TEST",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: fill in per requirements.json (lean rule: load program goes in as attached Excel file)
        # "marker_loads_active": False,
        # "stop_reason": "FRACTURE",  # or RUN_OUT / CRACK (FATIGUE_STOP_REASON)
    },
)
cyclic_fatigue_test.save()
print(f"CyclicFatigueTest created: {cyclic_fatigue_test.permId}")

# 11) FatigueDataEvaluation — EXPERIMENTAL_STEP.FATIGUE_DATA_EVALUATION  (§5.3.11)
fat_eval = o.new_sample(
    type="EXPERIMENTAL_STEP.FATIGUE_DATA_EVALUATION",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: fill in per requirements.json
        # "cycles_gross_final": 1_234_567,
        # "cycles_net_final": 1_200_000,
        # "cycles_dms_deviation_10": "DMS1: 1.0e6; DMS2: 1.1e6",
    },
)
fat_eval.save()
print(f"FatigueDataEvaluation created: {fat_eval.permId}")

# 12) FractureSurfaceAnalysis — EXPERIMENTAL_STEP.FRACTURE_SURFACE_ANALYSIS  (§5.3.12)
frac_surf = o.new_sample(
    type="EXPERIMENTAL_STEP.FRACTURE_SURFACE_ANALYSIS",
    space=SPACE,
    experiment=COLLECTION_ID,
    parents=[specimen.permId],
    props={
        # TODO: object_link to Camera (REUSE) and digital microscope Instrument (REUSE)
    },
)
frac_surf.save()
print(f"FractureSurfaceAnalysis created: {frac_surf.permId}")

In [ ]:
# Dataset upload stubs — attach raw/analyzed files to the relevant ExperimentalStep.
# Example 1: attach the load program (Excel) to CyclicFatigueTest.
# ds = o.new_dataset(
#     type="ANALYZED_DATA",
#     experiment=COLLECTION_ID,
#     sample=cyclic_fatigue_test.permId,
#     files=["path/to/load_program.xlsx"],  # TODO: replace
#     props={},
#     kind="PHYSICAL",
# )
# ds.save()
# print(f"Dataset uploaded (load program): {ds.permId}")

# Example 2: attach the installation-stress measurement file.
# ds_inst = o.new_dataset(
#     type="RAW_DATA",
#     experiment=COLLECTION_ID,
#     sample=inst_stress.permId,
#     files=["path/to/installation_stress.csv"],  # TODO: replace
#     props={},
#     kind="PHYSICAL",
# )
# ds_inst.save()
# print(f"Dataset uploaded (installation stress): {ds_inst.permId}")

In [ ]:
o.logout()
print("Session closed.")

## Using parsers for ongoing data ingestion

This notebook (cells 1-12) is the **one-time setup** path: it creates the Space/Project/Collection hierarchy, the parent specimen Object, and the ExperimentalStep skeleton.

For **per-run ingestion** (pushing new ExperimentalStep instances, properties, and DataSets after each experiment), use the generated parser package — this is the second of the two-path model:

```bash
pip install -e generated/welded_fatigue/parsers/
```

Then invoke the parser via `run_parser()` (see the stub in the next cell). The parser handles collection lookup/creation, sample creation, and dataset attachment automatically based on the parsed source files.

In [ ]:
# Parser usage stub (per-run ingestion path).
# TODO: pip install -e generated/welded_fatigue/parsers/ first
# TODO: replace WeldedFatigueParser import path and input file paths
#
# from welded_fatigue_parser import WeldedFatigueParser
# from bam_masterdata.cli.run_parser import run_parser
#
# run_parser(
#     openbis=o,
#     space_name=SPACE,
#     project_name=PROJECT_CODE,
#     collection_name=COLLECTION_CODE,
#     files_parser={WeldedFatigueParser(): ["path/to/your_data.csv"]},  # TODO: replace path
#     collection_type="DEFAULT_EXPERIMENT",
# )